# 🧠 Framing the Titanic Problem (as a Data Scientist)

## Understanding the Problem

The Titanic dataset presents a **binary classification** problem:

> Can we predict whether a passenger survived the Titanic disaster, based on their demographic and travel information?

This challenge is based on historical data from the RMS Titanic tragedy in 1912. The dataset includes features such as:

- **Pclass**: Ticket class (1st, 2nd, 3rd)
- **Sex**, **Age**
- **SibSp**, **Parch**: Number of relatives aboard
- **Fare**
- **Embarked**: Port of embarkation
- **Name**, **Ticket**, **Cabin**: May hold useful information like titles or patterns

We are provided with labeled training data (`Survived`) and asked to build a model that can predict survival on unseen test data.

---

## Why Use Machine Learning?

Machine learning is well-suited for this problem because:

-  **Labeled historical data** is available for training
-  Relationships between features and survival may be **nonlinear and complex**
-  ML models can **automatically detect patterns** in the data
-  A trained model is **reusable and scalable** for new or hypothetical passenger data

This problem mimics real-world classification use cases like:
- Predicting customer churn
- Credit default risk
- Disease diagnosis

---

## Evaluation Strategy

The competition uses **Accuracy** as the main metric:

> `Accuracy = (Correct Predictions) / (Total Predictions)`

While this is appropriate here, as data scientists we should also consider:

-  **Cross-validation** (e.g., 5-fold) for more reliable performance estimates
-  **Confusion Matrix** for understanding prediction distribution


### Good Practices:

- Use a **training/validation split** (or cross-validation) to select models
- Track not only accuracy, but also **precision**, **recall**, and **F1-score** if needed

---

## Summary

The Titanic challenge is a classic and practical machine learning task. It provides an excellent sandbox to:

- Apply classification algorithms
- Practice data cleaning and feature engineering
- Learn evaluation techniques
- Understand generalization and overfitting

Our goal is to build a model that performs **well on unseen data**, not just the training set.



# Data Exploration

Before we dive into preprocessing and modeling, we need to understand the dataset.

This step helps us identify:
- Data types (numerical vs. categorical)
- Missing values (e.g., Age, Cabin, Embarked)
- Outliers or unusual distributions
- Potential relationships between variables and the target

By running `.info()`, `.describe()`, and `.head()`, we get a high-level overview of the dataset's structure, quality, and quirks. This exploration informs our decisions during feature engineering and cleaning.

---


In [1]:
# Load data
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Quick look at the training data
train_df.head()
train_df.info()
train_df.describe(include='all')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Dooley, Mr. Patrick",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


# Feature Engineering

Feature engineering helps the model learn patterns that may not be obvious from raw data.

Key new features:
- `Title`: extracted from the passenger's name using a regular expression ` ([A-Za-z]+)\.`. This captures honorifics like Mr, Miss, Dr, etc., which correlate with age, gender, and class.
- `FamilySize`: calculated as `SibSp + Parch + 1` to include the passenger. Larger families may face challenges surviving together, while solo travelers (`IsAlone = 1`) may behave differently in emergencies.
- `IsAlone`: binary flag for passengers traveling alone. Traveling with family often affects survival chances.

These engineered features inject domain knowledge into the model and often improve predictive performance.

---


In [3]:
# Combine train and test for uniform preprocessing
import numpy as np

test_df['Survived'] = np.nan
full_df = pd.concat([train_df, test_df], sort=False)

# Extract title from name
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
full_df['Title'] = full_df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
                                             'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
full_df['Title'] = full_df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Family size
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1
full_df['IsAlone'] = (full_df['FamilySize'] == 1).astype(int)

# Drop unused columns
full_df = full_df.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1)



# Data Cleaning

Data cleaning ensures that the dataset is complete and consistent before we feed it into a machine learning model.

Key actions in this step:
- Fill missing values:
  - `Embarked`: fill with mode (most common value)
  - `Fare`: fill using median of the corresponding passenger class
  - `Age`: fill with group-wise medians based on `Title` and `Pclass` (a more informed guess than global median)
- Remove noisy or high-cardinality features:
  - `Cabin`: too many missing values
  - `Ticket` and `Name`: complex or redundant
  - `PassengerId`: not useful for prediction

Cleaning improves model robustness and avoids errors caused by missing data or irrelevant features.

---


In [4]:


# Fill missing values
full_df['Embarked'] = full_df['Embarked'].fillna(full_df['Embarked'].mode()[0])
full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))


---

## ⚠️ Leakage Audit — Problem Identified Here

### Where the leakage happens

The three imputation lines in the cell above are the source of data leakage.

At this point in the notebook, `full_df` contains **both** the training set (891 rows) and the test set (418 rows) — they were concatenated in the Feature Engineering cell. When we call `.mode()[0]` or `.median()` inside a `transform`, pandas computes those statistics over the **entire combined dataset**, not just the training rows.

Concretely:

| Line | Statistic computed on |
|------|-----------------------|
| `Embarked` → `mode()[0]` | 1 307 non-null rows (train + test) |
| `Fare` → `median()` per Pclass | fares from both sets |
| `Age` → `median()` per Title+Pclass | ages from both sets |

Those fill values then flow back into the **training rows**, which means the training set is now partially shaped by test-set distributions.

### Why this violates the train/test separation principle

A model trained on this data has implicitly "seen" the test set during preprocessing. Even though no `Survived` labels leak, statistical properties of the test set (its age distribution, fare levels, etc.) influence how training gaps are filled. This makes cross-validation scores optimistic and can cause the model to underperform on genuinely unseen data.

### What the fix looks like

Compute all imputation statistics on the **training set only**, store the resulting values, and use them to fill missing entries in both train and test. The `SimpleImputer`, `StandardScaler`, and `OneHotEncoder` further down already do this correctly — the fix brings this earlier step in line with that pattern.

> Fix implemented in branch `fix/data-leakage`.

---

#  Preprocessing – Numeric Features

Most ML models expect numeric inputs, and some are sensitive to feature scales.

In this step:
- **Imputation**: Although we already filled many values earlier, we run a final imputation step (with median) to catch any remaining NaNs.
- **Scaling**: We standardize numeric features using `StandardScaler`, which transforms them to have mean = 0 and standard deviation = 1.

This helps models that rely on distance metrics (e.g., logistic regression, SVMs) and ensures features contribute equally to the model.

---
# Preprocessing – Categorical Features

Machine learning models can't work with text labels directly. We use **One-Hot Encoding** to convert categorical variables into binary indicator columns.

- Each category (e.g., `Embarked = S`) becomes its own column (`Embarked_S`, `Embarked_C`, etc.).
- We use `handle_unknown='ignore'` to prevent errors if the test set includes categories not seen during training.

This step makes categorical variables model-friendly while preserving their distinct values.

---

# Feature Assembly

We now combine the preprocessed numeric and categorical features to form a single dataset for training.

- Numeric features are imputed and scaled.
- Categorical features are one-hot encoded.
- We concatenate them horizontally using NumPy to create the final feature matrix.

This is the final input format the model will learn from.

---


In [6]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate train/test
train_df = full_df[full_df['Survived'].notnull()]
test_df = full_df[full_df['Survived'].isnull()].drop('Survived', axis=1)

X = train_df.drop('Survived', axis=1)
y = train_df['Survived'].astype(int)

# Select feature types
categorical_cols = ['Sex', 'Embarked', 'Title']
numeric_cols = ['Age', 'Fare', 'FamilySize', 'IsAlone', 'Pclass', 'SibSp', 'Parch']

# Numeric preprocessing
num_imputer = SimpleImputer(strategy='median')
X_numeric = num_imputer.fit_transform(X[numeric_cols])
test_numeric = num_imputer.transform(test_df[numeric_cols])

scaler = StandardScaler()
X_numeric = scaler.fit_transform(X_numeric)
test_numeric = scaler.transform(test_numeric)

# Categorical preprocessing
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
X_categorical = encoder.fit_transform(X[categorical_cols])
test_categorical = encoder.transform(test_df[categorical_cols])

# Merge processed data
from numpy import hstack
X_final = hstack([X_numeric, X_categorical])
test_final = hstack([test_numeric, test_categorical])


# Model Training & Evaluation

We use a **Random Forest Classifier**, a tree-based ensemble model that performs well on tabular data and handles both categorical and numerical features effectively.

- We train the model using the full preprocessed dataset.
- We evaluate its generalization using **5-fold cross-validation**, which splits the training data into 5 parts and tests the model on each.

Cross-validation gives a more reliable estimate of real-world performance compared to a single train/test split.

---



In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

# Cross-validation to evaluate generalization
cv_scores = cross_val_score(clf, X_final, y, cv=5, scoring='accuracy')
print(f"Cross-validated accuracy: {cv_scores.mean():.4f}")

# Train on full data
clf.fit(X_final, y)


Cross-validated accuracy: 0.8215


RandomForestClassifier(max_depth=6, random_state=42)

# 📦 Inference and Submission

This is the final step of the ML pipeline.

- We apply the trained model to the (unlabeled) test data to generate survival predictions.
- We format the results according to Kaggle's submission rules: a CSV file with `PassengerId` and `Survived` columns.
- We save the predictions using `to_csv`.

This stage represents how a real model would be used in practice: taking new data, making predictions, and outputting actionable results.

---



In [10]:
# Predict on test data
predictions = clf.predict(test_final)

# Create submission
submission = pd.DataFrame({
    'PassengerId': test_df.index + 892,  # Assuming test.csv starts at ID 892
    'Survived': predictions.astype(int)
})
submission.to_csv('titanic_submission.csv', index=False)
print("✅ Submission file created: titanic_submission.csv")


✅ Submission file created: titanic_submission.csv
